[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_3_value_iteration_complete.ipynb)

<div style="text-align:center">
    <h1>
        Value Iteration
    </h1>
</div>
<br>

<div style="text-align:center">
    <p>
        In this notebook we are going to look at a dynamic programming algorithm called value iteration. In it, we will sweep the state space and update all the V(s) values.
    </p>
</div>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_policy, plot_values


## Import the necessary software libraries:

In [ ]:
# NumPy gives us fast array math for the value table and policy.
import numpy as np
# Matplotlib is used to draw the maze, the policy arrows and the value heatmap.
import matplotlib.pyplot as plt

## Initialize the environment

In [ ]:
# Create the 5x5 grid-world maze. `env` knows the rules: how actions move the
# agent, what reward each step gives, and where the goal is.
env = Maze()

In [ ]:
# Ask the environment for a picture (an RGB image array) of the current maze.
frame = env.render(mode='rgb_array')
# Make the figure a comfortable 6x6 inches.
plt.figure(figsize=(6,6))
# Hide the x/y axis ticks so we only see the maze itself.
plt.axis('off')
# Display the maze image.
plt.imshow(frame)

In [ ]:
# The observation (state) is the agent's grid cell: a (row, col) pair, each 0-4.
print(f"Observation space shape: {env.observation_space.nvec}")
# There are 4 possible actions (up, down, left, right).
print(f"Number of actions: {env.action_space.n}")

## Define the policy $\pi(\cdot|s)$

#### Create the policy $\pi(\cdot|s)$

In [ ]:
# The policy stores, for every state, the probability of choosing each action.
# Shape (5, 5, 4): one row, one column, and 4 action-probabilities per cell.
# We start with 0.25 everywhere, i.e. a uniform random policy (each action equally likely).
policy_probs = np.full((5, 5, 4), 0.25)

In [ ]:
# A small helper: given a state (row, col), return its 4 action probabilities.
def policy(state):
    return policy_probs[state]

#### Test the policy with state (0, 0)

In [ ]:
# Look up the action probabilities the policy assigns at the top-left cell (0, 0).
action_probabilities = policy((0,0))
# Print each action's probability; for the random policy they should all be 0.25.
for action, prob in zip(range(4), action_probabilities):
    print(f"Probability of taking action {action}: {prob}")

#### See how the random policy does in the maze

In [ ]:
# Run the (still random) policy in the maze for one episode and animate it,
# so we can see how badly an untrained agent wanders around.
test_agent(env, policy, episodes=1)

#### Plot the policy

In [ ]:
# Draw the policy as arrows on top of the maze image.
plot_policy(policy_probs, frame)

## Define value table $V(s)$

#### Create the $V(s)$ table

In [ ]:
# The value table V(s): one number per cell estimating how good it is to be there
# (the expected total future reward). We start every cell at 0.
state_values = np.zeros(shape=(5,5))

#### Plot V(s)

In [ ]:
# Draw the value table as a heatmap on top of the maze (all zeros for now).
plot_values(state_values, frame)

## Implement the Value Iteration algorithm

**The big idea (in plain language).**
Value iteration is a shortcut. Instead of fully evaluating a policy and then improving it
(as policy iteration does), it folds both steps into one. In every sweep over the grid we set
each state's value to the value of its *best* action, not the average over actions.

For each state we:
1. try all 4 actions and compute `Q(s, a) = reward + gamma * V(next_state)` for each;
2. keep the largest `Q` (the best action) and store it as the state's new value;
3. remember which action was best - that becomes the policy for this state.

Because we always grab the best action, the values steadily climb toward the *optimal* values.
We keep sweeping until no value changes by more than a tiny amount (`theta`), at which point
both the values and the greedy policy they imply are optimal.

Symbols used below:
- `gamma` (the discount factor, 0.99) makes sooner rewards worth slightly more than later ones.
- `theta` is the small stopping threshold for convergence.
- `Q(s, a)` (called `qsa`) is the value of taking action `a` now and acting well afterwards.
- The `max` over actions is the key difference from policy evaluation's probability-weighted average.

</br>

<div style="text-align:center">
    Adapted from Barto & Sutton: "Reinforcement Learning: An Introduction".
</div>

In [ ]:
# VALUE ITERATION: find the optimal values directly, without a separate policy first.
# In each sweep we set every state's value to the best action's value (a max, not an average).
def value_iteration(policy_probs, state_values, theta=1e-6, gamma=0.99):
    # `delta` tracks the biggest change to any value in a sweep; start it huge so we enter the loop.
    delta = float('inf')

    # Keep sweeping while some value still changed by more than the tiny threshold theta.
    while delta > theta:
        # Reset the largest-change tracker at the start of each full sweep.
        delta = 0
        # Visit every cell of the 5x5 grid.
        for row in range(5):
            for col in range(5):
                # Remember the current value so we can measure how much it changes.
                old_value = state_values[(row, col)]
                # We will record the greedy (best) action here as a one-hot probability vector.
                action_probs = None
                # Track the value of the best action found so far in this state.
                max_qsa = float('-inf')

                # Try each of the 4 actions and score it.
                for action in range(4):
                    # Simulate the action to see the next state and the reward.
                    next_state, reward, _, _ = env.simulate_step((row, col), action)
                    # Q(s,a): immediate reward plus discounted value of where we land.
                    qsa = reward + gamma * state_values[next_state]
                    # If this action beats the best so far, keep it and remember it as the greedy choice.
                    if qsa > max_qsa:
                        max_qsa = qsa
                        action_probs = np.zeros(4)
                        action_probs[action] = 1.

                # The new value of the state is the value of its best action (the Bellman optimality update).
                state_values[(row, col)] = max_qsa
                # Record that best action as this state's policy.
                policy_probs[(row, col)] = action_probs

                # Track the largest value change seen this sweep (used to decide when to stop).
                delta = max(delta, abs(max_qsa - old_value))

In [ ]:
# Run the algorithm. It fills in the optimal `state_values` and the greedy `policy_probs` in place.
value_iteration(policy_probs, state_values)

## Show results

#### Show resulting value table $V(s)$

In [ ]:
# Show the final value heatmap: cells near the goal should now have higher values.
plot_values(state_values, frame)

#### Show resulting policy $\pi(\cdot|s)$

In [ ]:
# Show the final policy: arrows should now point along the shortest path to the goal.
plot_policy(policy_probs, frame)

#### Test the resulting agent

In [ ]:
# Run the optimal policy in the maze to watch it solve the task efficiently.
test_agent(env, policy)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch. 4: Dynamic Programming](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)